# Clustering Distrital con Grafos — VGAE Temporal

Se construye **un solo grafo temporal** con todos los años (2020–2025):
- **Nodos** = distrito × año (~11,334 nodos)
- **Aristas espaciales** = contigüidad Queen entre distritos del mismo año
- **Aristas temporales** = mismo distrito entre años consecutivos

Se entrena **un solo VGAE**. Al venir todos los embeddings del mismo espacio latente, los clusters son comparables entre años.

## 0. Dependencias

In [ ]:
# !pip install torch-geometric libpysal umap-learn

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.impute import SimpleImputer

import torch
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import GCNConv, VGAE
from torch_geometric.utils import from_scipy_sparse_matrix
import scipy.sparse as sp

import libpysal
import umap

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.manual_seed(42)
print('Device:', DEVICE)

## 1. Cargar datos

In [ ]:
PANEL_PATH   = '../../data/clean/merged/panel_distrital_clean.csv'
GEOJSON_PATH = '../03_causal_model/cghciv.geojson'

panel = pd.read_csv(PANEL_PATH)
gdf   = gpd.read_file(GEOJSON_PATH)
gdf['UBIGEO'] = gdf['UBIGEO'].astype(str).str.zfill(6)
gdf_sorted    = gdf.sort_values('UBIGEO').reset_index(drop=True)
GEO_UBIGEOS   = gdf_sorted['UBIGEO'].tolist()

YEARS = sorted(panel['anio'].unique().tolist())
N_DISTRICTS = len(GEO_UBIGEOS)
N_YEARS     = len(YEARS)

print(f'Distritos: {N_DISTRICTS} | Años: {YEARS}')
print(f'Nodos totales en el grafo temporal: {N_DISTRICTS * N_YEARS}')

## 2. Features de nodo

In [ ]:
NODE_FEATURES = [
    'prevalencia_anemia',
    'gasto_total', 'gasto_anemia_pan',
    'personal_total', 'programa_anemia', 'centro_salud_municipal',
    'altitude', 'superficie', 'pob_densidad_2020',
    'pct_cultivo', 'pct_construido', 'pct_desnudo', 'pct_agua_visible',
    'n_edificios', 'densidad_edificios_km2',
    'elevacion_media', 'pendiente_media',
    'pct_agua_permanente', 'pct_agua_estacional',
]
print(f'Features: {len(NODE_FEATURES)}')

## 3. Construcción del grafo temporal

El nodo `i` en el año `t` tiene índice global `t * N_DISTRICTS + i`.

- **Aristas espaciales**: Queen contiguity dentro de cada año (replicadas 6 veces)
- **Aristas temporales**: mismo distrito entre años consecutivos

In [ ]:
# --- Aristas espaciales (Queen) ---
w_queen = libpysal.weights.Queen.from_dataframe(gdf_sorted, silence_warnings=True)
sp_queen = w_queen.sparse  # (N_DISTRICTS × N_DISTRICTS)

spatial_edges = []
for t in range(N_YEARS):
    offset = t * N_DISTRICTS
    cx = sp_queen.tocoo()
    src = cx.row + offset
    dst = cx.col + offset
    spatial_edges.append(np.stack([src, dst]))

# --- Aristas temporales (mismo distrito, años consecutivos) ---
temporal_edges = []
for t in range(N_YEARS - 1):
    idx = np.arange(N_DISTRICTS)
    src = idx + t * N_DISTRICTS
    dst = idx + (t + 1) * N_DISTRICTS
    # bidireccional
    temporal_edges.append(np.stack([src, dst]))
    temporal_edges.append(np.stack([dst, src]))

all_edges = np.concatenate(spatial_edges + temporal_edges, axis=1)
EDGE_INDEX = torch.tensor(all_edges, dtype=torch.long)

print(f'Aristas espaciales : {sum(e.shape[1] for e in spatial_edges):,}')
print(f'Aristas temporales : {sum(e.shape[1] for e in temporal_edges):,}')
print(f'Total aristas      : {EDGE_INDEX.shape[1]:,}')

## 4. Construcción de la matriz de features X

In [ ]:
blocks = []
node_meta = []  # (ubigeo, anio) por cada nodo

for year in YEARS:
    df_yr = panel[panel['anio'] == year].copy()
    df_yr['ubigeo_str'] = df_yr['ubigeo'].astype(str).str.zfill(6)
    df_yr = df_yr.set_index('ubigeo_str').reindex(GEO_UBIGEOS)
    X = df_yr[NODE_FEATURES].values.astype(np.float32)
    blocks.append(X)
    node_meta.extend([(u, year) for u in GEO_UBIGEOS])

X_all = np.vstack(blocks)  # (N_DISTRICTS * N_YEARS, n_features)

# Imputar y normalizar globalmente
X_all = SimpleImputer(strategy='median').fit_transform(X_all)
X_all = StandardScaler().fit_transform(X_all)

x = torch.tensor(X_all, dtype=torch.float)
data = Data(x=x, edge_index=EDGE_INDEX)

node_meta_df = pd.DataFrame(node_meta, columns=['ubigeo', 'anio'])

print(f'X shape: {X_all.shape}')
print(f'Grafo: {data.num_nodes} nodos, {data.num_edges} aristas, {data.num_node_features} features')

## 5. Modelo VGAE

In [ ]:
class VGAEEncoder(torch.nn.Module):
    def __init__(self, in_channels, hidden=128, out=64):
        super().__init__()
        self.conv_shared = GCNConv(in_channels, hidden)
        self.conv_mu     = GCNConv(hidden, out)
        self.conv_logstd = GCNConv(hidden, out)

    def forward(self, x, edge_index):
        h = F.relu(self.conv_shared(x, edge_index))
        return self.conv_mu(h, edge_index), self.conv_logstd(h, edge_index)

## 6. Entrenamiento

In [ ]:
EPOCHS = 400
LR     = 1e-2

data  = data.to(DEVICE)
model = VGAE(VGAEEncoder(data.num_node_features)).to(DEVICE)
opt   = torch.optim.Adam(model.parameters(), lr=LR)

model.train()
for epoch in range(1, EPOCHS + 1):
    opt.zero_grad()
    z    = model.encode(data.x, data.edge_index)
    loss = model.recon_loss(z, data.edge_index) + (1 / data.num_nodes) * model.kl_loss()
    loss.backward()
    opt.step()
    if epoch % 100 == 0:
        print(f'epoch {epoch:4d} | loss {loss.item():.4f}')

model.eval()
with torch.no_grad():
    Z = model.encode(data.x, data.edge_index).cpu().numpy()

print(f'\nEmbeddings Z: {Z.shape}')

## 7. Clustering global

In [ ]:
# Buscar k óptimo sobre todos los embeddings
sil_scores = {}
for k in range(3, 12):
    labels_k = KMeans(n_clusters=k, random_state=42, n_init=10).fit_predict(Z)
    sil_scores[k] = silhouette_score(Z, labels_k)

best_k = max(sil_scores, key=sil_scores.get)
print('Silhouette por k:', {k: round(v, 3) for k, v in sil_scores.items()})
print(f'Mejor k: {best_k} (silhouette={sil_scores[best_k]:.3f})')

LABELS = KMeans(n_clusters=best_k, random_state=42, n_init=10).fit_predict(Z)
node_meta_df['cluster'] = LABELS

## 8. Visualización: UMAP

In [ ]:
reducer = umap.UMAP(n_components=2, random_state=42)
Z2d = reducer.fit_transform(Z)
node_meta_df['umap_x'] = Z2d[:, 0]
node_meta_df['umap_y'] = Z2d[:, 1]

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
for i, year in enumerate(YEARS):
    mask = node_meta_df['anio'] == year
    sub  = node_meta_df[mask]
    axes.flatten()[i].scatter(
        sub['umap_x'], sub['umap_y'],
        c=sub['cluster'], cmap='tab10',
        vmin=0, vmax=best_k - 1,
        s=6, alpha=0.7
    )
    axes.flatten()[i].set_title(str(year))
    axes.flatten()[i].axis('off')

plt.suptitle(f'UMAP — embeddings VGAE temporal ({best_k} clusters, mismo espacio latente)', fontsize=13)
plt.tight_layout()
plt.savefig('fig_umap_clusters.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Visualización: mapas coropléticos (colores consistentes entre años)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 12))

for i, year in enumerate(YEARS):
    mask     = node_meta_df['anio'] == year
    clusters = node_meta_df.loc[mask, 'cluster'].values

    gdf_plot = gdf_sorted.copy()
    gdf_plot['cluster'] = clusters
    gdf_plot.plot(
        column='cluster', ax=axes.flatten()[i],
        cmap='tab10', vmin=0, vmax=best_k - 1,
        linewidth=0.1, edgecolor='white'
    )
    axes.flatten()[i].set_title(str(year))
    axes.flatten()[i].axis('off')

plt.suptitle(f'Clusters distritales — VGAE temporal ({best_k} clusters)\nMismo color = mismo cluster en todos los años', fontsize=12)
plt.tight_layout()
plt.savefig('fig_map_clusters.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. Perfil epidemiológico por cluster

In [ ]:
df_full = panel.copy()
df_full['ubigeo_str'] = df_full['ubigeo'].astype(str).str.zfill(6)

df_full = df_full.merge(
    node_meta_df.rename(columns={'ubigeo': 'ubigeo_str'}),
    on=['ubigeo_str', 'anio'],
    how='left'
)

profile_cols = [
    'prevalencia_anemia', 'gasto_total', 'gasto_anemia_pan',
    'elevacion_media', 'pob_densidad_2020', 'pendiente_media',
]

profile = df_full.groupby('cluster')[profile_cols].mean().round(3)
profile['n_obs'] = df_full.groupby('cluster').size()
profile['macroregion_top'] = df_full.groupby('cluster')['macroregion_inei'].agg(
    lambda x: x.value_counts().index[0] if x.notna().any() else '?'
)

pd.set_option('display.float_format', lambda x: f'{x:,.2f}')
print(profile.to_string())

## 11. Exportar

In [ ]:
out = node_meta_df[['ubigeo', 'anio', 'cluster']].copy()
out.to_csv('../../outputs/clusters_distritales.csv', index=False)
print('Guardado: outputs/clusters_distritales.csv')
out.head()